# Week 01 — YOLO Smoke Test on VisDrone

**Run ID:** `SMOKE-001`  
**Purpose:** verify that the Kaggle GPU, Ultralytics YOLO, VisDrone input, inference, visualization, and result-export pipeline work end to end.

> This is a qualitative smoke test with COCO-pretrained weights. It is not a VisDrone baseline evaluation, and its predictions must not be reported as mAP results.

## 1. Install Ultralytics

Run this cell whenever Kaggle starts a fresh session.

In [ ]:
%pip install -q -U ultralytics

## 2. Verify the environment

In [ ]:
import json
import platform
import random
import shutil
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from PIL import Image
from ultralytics import YOLO

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")

assert torch.cuda.is_available(), "GPU is not enabled. Enable a GPU accelerator in Kaggle settings."

## 3. Set the experiment configuration

The smoke test uses one T4 GPU. The second GPU is not needed for a small inference run.

In [ ]:
SEED = 42
MODEL_NAME = "yolo11n.pt"
IMAGE_SIZE = 640
CONFIDENCE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.70
SAMPLE_COUNT = 8
DEVICE = 0

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/results/smoke-test")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Output directory:", OUTPUT_ROOT)

## 4. Find or download VisDrone validation images

The next cell first searches `/kaggle/input` for an attached VisDrone dataset. If none is found, it downloads only the official VisDrone2019-DET validation archive used by Ultralytics. Kaggle Internet access must be enabled for the automatic-download fallback.

In [ ]:
from ultralytics.utils import ASSETS_URL
from ultralytics.utils.downloads import download

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
DOWNLOAD_ROOT = Path("/kaggle/working/datasets/VisDrone")

def find_visdrone_images(root):
    if not root.exists():
        return []
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.lower() in VALID_EXTENSIONS
        and "visdrone" in str(path).lower()
    )

visdrone_images = find_visdrone_images(INPUT_ROOT)

if not visdrone_images:
    DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
    validation_url = f"{ASSETS_URL}/VisDrone2019-DET-val.zip"
    print("No attached VisDrone input found. Downloading the validation split...")
    download([validation_url], dir=DOWNLOAD_ROOT, threads=1)
    visdrone_images = find_visdrone_images(DOWNLOAD_ROOT)

if not visdrone_images:
    raise FileNotFoundError(
        "The VisDrone validation archive was not found or extracted. "
        "Confirm that Kaggle Internet access is enabled, then rerun this cell."
    )

sample_size = min(SAMPLE_COUNT, len(visdrone_images))
sample_indices = np.linspace(0, len(visdrone_images) - 1, sample_size, dtype=int)
sample_images = [visdrone_images[index] for index in sample_indices]

print(f"Found {len(visdrone_images):,} VisDrone images.")
print(f"Selected {len(sample_images)} images for SMOKE-001:")
for path in sample_images:
    print(" -", path)

## 5. Inspect sample images

This verifies that the input images can be opened and gives an initial view of resolution, density, scale variation, and occlusion.

In [ ]:
image_metadata = []

for path in sample_images:
    with Image.open(path) as image:
        width, height = image.size
    image_metadata.append(
        {"file_name": path.name, "width": width, "height": height, "path": str(path)}
    )

display(pd.DataFrame(image_metadata))

In [ ]:
preview_count = min(4, len(sample_images))
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for axis, path in zip(axes, sample_images[:preview_count]):
    with Image.open(path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(path.name)
    axis.axis("off")

for axis in axes[preview_count:]:
    axis.axis("off")

plt.tight_layout()
plt.show()

## 6. Load the pretrained YOLO model

The model will download automatically the first time it is used.

In [ ]:
model = YOLO(MODEL_NAME)
print(f"Loaded {MODEL_NAME}")

## 7. Run pretrained inference

Predictions are saved under `/kaggle/working/results/smoke-test/predictions`.

In [ ]:
results = model.predict(
    source=[str(path) for path in sample_images],
    imgsz=IMAGE_SIZE,
    conf=CONFIDENCE_THRESHOLD,
    iou=IOU_THRESHOLD,
    device=DEVICE,
    save=True,
    project=str(OUTPUT_ROOT),
    name="predictions",
    exist_ok=True,
    verbose=False,
)

print(f"Inference completed for {len(results)} images.")
print("Saved predictions to:", results[0].save_dir)

## 8. Summarize detections and speed

In [ ]:
summary_rows = []

for source_path, result in zip(sample_images, results):
    class_ids = [] if result.boxes is None else result.boxes.cls.int().cpu().tolist()
    class_names = [result.names[class_id] for class_id in class_ids]
    class_counts = dict(Counter(class_names))

    summary_rows.append(
        {
            "file_name": source_path.name,
            "detections": len(class_ids),
            "classes": json.dumps(class_counts, sort_keys=True),
            "preprocess_ms": round(result.speed.get("preprocess", 0.0), 3),
            "inference_ms": round(result.speed.get("inference", 0.0), 3),
            "postprocess_ms": round(result.speed.get("postprocess", 0.0), 3),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_ROOT / "smoke_test_summary.csv"
summary_df.to_csv(summary_path, index=False)

display(summary_df)
print("Mean inference time (ms/image):", round(summary_df["inference_ms"].mean(), 3))
print("Summary saved to:", summary_path)

## 9. Visualize predictions

Look for missed tiny objects, false positives, dense regions, and differences between large and small objects.

In [ ]:
preview_count = min(4, len(results))
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for axis, result in zip(axes, results[:preview_count]):
    annotated_bgr = result.plot()
    annotated_rgb = annotated_bgr[..., ::-1]
    axis.imshow(annotated_rgb)
    axis.set_title(Path(result.path).name)
    axis.axis("off")

for axis in axes[preview_count:]:
    axis.axis("off")

plt.tight_layout()
plt.show()

## 10. Save reproducibility metadata

In [ ]:
run_metadata = {
    "run_id": "SMOKE-001",
    "python_version": sys.version.split()[0],
    "pytorch_version": torch.__version__,
    "ultralytics_version": ultralytics.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_names": [
        torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())
    ],
    "model": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "device_used": DEVICE,
    "sample_count": len(sample_images),
    "seed": SEED,
}

metadata_path = OUTPUT_ROOT / "run_metadata.json"
with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)

print(json.dumps(run_metadata, indent=2))
print("Metadata saved to:", metadata_path)

## 11. Export the smoke-test results

The ZIP file can be downloaded from the Kaggle output panel. Do not commit the full archive or model weights to GitHub.

In [ ]:
archive_path = shutil.make_archive(
    "/kaggle/working/week-01-yolo-smoke-test-results",
    "zip",
    root_dir=OUTPUT_ROOT,
)
print("Created:", archive_path)

## 12. Manual observations

Complete these notes after reviewing the predictions:

- **Images tested:** 8 VisDrone validation images.
- **Objects detected reasonably well:** Medium and large cars, buses, and trucks were detected more consistently than small objects.
- **Small objects frequently missed:** Distant pedestrians, motorcycles, bicycles, and small vehicles were frequently missed.
- **False positives:** Some objects were incorrectly classified as traffic lights or potted plants.
- **Dense or occluded regions:** The model detected several cars in dense scenes but missed many pedestrians and smaller road users.
- **Effect of resizing to 640 pixels:** Distant objects became extremely small after resizing, which likely reduced the visual information available to the detector.
- **Most important failure case:** One overexposed image produced only one detection even though many vehicles and people were visible.
- **Initial hypothesis about image tiling:** Image tiling may make small objects larger relative to the model input and improve detection, but this must later be tested using a model trained on VisDrone.

### Completion rule

`SMOKE-001` is complete when inference succeeds, prediction images are visible, `smoke_test_summary.csv` and `run_metadata.json` exist, and the observations above are written in your own words.